In [1]:
# The DAG Executor
# Given a task dict mapping task names to {'deps': [dep_names], 'func': lambda_source_string}, topologically order the tasks. For each task, evaluate its lambda by passing a dict {dep_name: result_of_dep} as 'inputs'. Return a dict of task_name -> result. Raise an exception if a cycle exists.
# EXAMPLE
# INPUT
# tasks
# :
# {
#   "load": {
#     "deps": ["transform"],
#     "func": "lambda inputs: sum(inputs['transform'])"
#   },
#   "extract": {"deps":[],"func":"lambda inputs: [1, 2, 3]"},
#   "transform": {
#     "deps": ["extract"],
#     "func": "lambda inputs: [x * 2 for x in inputs['extract']]"
#   }
# }
# OUTPUT
# {"load":12,"extract":[1,2,3],"transform":[2,4,6]}


In [14]:
input_data={
  "load": {
    "deps": ["transform"],
    "func": "lambda inputs: sum(inputs['transform'])"
  },
  "extract": {"deps":[],"func":"lambda inputs: [1, 2, 3]"},
  "transform": {
    "deps": ["extract"],
    "func": "lambda inputs: [x * 2 for x in inputs['extract']]"
  }
}

In [23]:
def execute_dag(tasks):
    results = {}
    state = {}

    def run(task_name):
        if state.get(task_name) == 'visiting':       # loop? error
            raise Exception("Cycle detected")
        if state.get(task_name) == 'done':           # already done? reuse
            return results[task_name]

        state[task_name] = 'visiting'                # mark: starting

        inputs = {}
        for dep in tasks[task_name]['deps']:         # run deps first
            inputs[dep] = run(dep)

        func = eval(tasks[task_name]['func'])        # make the function
        results[task_name] = func(inputs)            # run it, save answer
        state[task_name] = 'done'                    # mark: finished
        return results[task_name]

    for task_name in tasks:
        run(task_name)
    return results

In [24]:
execute_dag(input_data)

{'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}

NameError: name 'input_data' is not defined

In [26]:
from collections import defaultdict, deque

# =========================
# Input
# =========================
tasks = {
    "load": {
        "deps": ["transform"],
        "func": "lambda inputs: sum(inputs['transform'])"
    },
    "extract": {
        "deps": [],
        "func": "lambda inputs: [1, 2, 3]"
    },
    "transform": {
        "deps": ["extract"],
        "func": "lambda inputs: [x * 2 for x in inputs['extract']]"
    }
}

# =========================
# Step 1: Validate dependencies
# =========================
for task_name, task_info in tasks.items():
    for dep in task_info.get("deps", []):
        if dep not in tasks:
            raise ValueError(f"Missing dependency '{dep}' for task '{task_name}'")

# =========================
# Step 2: Build graph
# If task depends on dep:
# dep -> task
# =========================
graph = defaultdict(list)
indegree = {}

for task_name in tasks:
    indegree[task_name] = 0

for task_name, task_info in tasks.items():
    for dep in task_info.get("deps", []):
        graph[dep].append(task_name)
        indegree[task_name] += 1

# =========================
# Step 3: Topological sort
# =========================
queue = deque()

for task_name, degree in indegree.items():
    if degree == 0:
        queue.append(task_name)

topo_order = []

while queue:
    current_task = queue.popleft()
    topo_order.append(current_task)

    for next_task in graph[current_task]:
        indegree[next_task] -= 1

        if indegree[next_task] == 0:
            queue.append(next_task)

# =========================
# Step 4: Cycle detection
# =========================
if len(topo_order) != len(tasks):
    raise ValueError("Cycle detected in task dependencies")

# =========================
# Step 5: Execute tasks
# =========================
results = {}

for task_name in topo_order:
    deps = tasks[task_name].get("deps", [])

    inputs = {}

    for dep in deps:
        inputs[dep] = results[dep]

    func = eval(tasks[task_name]["func"])

    results[task_name] = func(inputs)

# =========================
# Output
# =========================
print("Topological order:", topo_order)
print("Results:", results)

Topological order: ['extract', 'transform', 'load']
Results: {'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}


In [28]:
tasks = {

    "load": {

        "deps": ["transform"],

        "func": "lambda inputs: sum(inputs['transform'])"

    },

    "extract": {

        "deps": [],

        "func": "lambda inputs: [1, 2, 3]"

    },

    "transform": {

        "deps": ["extract"],

        "func": "lambda inputs: [x * 2 for x in inputs['extract']]"

    }

}

In [30]:
# store final results here
results = {}

# 1. run extract first
inputs = {}
func = eval(tasks["extract"]["func"])
print(func)
results["extract"] = func(inputs)

print(results)

<function <lambda> at 0x10bc3d280>
{'extract': [1, 2, 3]}


In [45]:
func({eval(tasks['extract']['func'])})



[1, 2, 3]

In [46]:
tasks = {
    "extract": {
        "deps": [],
        "func": "lambda inputs: [1, 2, 3]"
    }
}

func_string = tasks["extract"]["func"]
print(func_string)

func = eval(func_string)

result = func({})

print(result)

lambda inputs: [1, 2, 3]
[1, 2, 3]


In [47]:
tasks = {
    "extract": {
        "deps": [],
        "func": "lambda inputs: [1, 2, 3]"
    }
}

func_string = tasks["extract"]["func"]
print(func_string)

func = eval(func_string)

result = func({})

print(result)

lambda inputs: [1, 2, 3]
[1, 2, 3]


In [52]:
inputs = {
    "extract": results["extract"]
}

func = eval(tasks["transform"]["func"])
results["transform"] = func(inputs)

print(results)

KeyError: 'transform'

In [54]:
tasks = {

    "load": {

        "deps": ["transform"],

        "func": "lambda inputs: sum(inputs['transform'])"

    },

    "extract": {

        "deps": [],

        "func": "lambda inputs: [1, 2, 3]"

    },

    "transform": {

        "deps": ["extract"],

        "func": "lambda inputs: [x * 2 for x in inputs['extract']]"

    }

}

In [55]:
tasks = {
    "load": {
        "deps": ["transform"],
        "func": "lambda inputs: sum(inputs['transform'])"
    },
    "extract": {
        "deps": [],
        "func": "lambda inputs: [1, 2, 3]"
    },
    "transform": {
        "deps": ["extract"],
        "func": "lambda inputs: [x * 2 for x in inputs['extract']]"
    }
}

In [56]:
# store final results here
results = {}

# 1. run extract first
inputs = {}
func = eval(tasks["extract"]["func"])
results["extract"] = func(inputs)

print(results)

{'extract': [1, 2, 3]}


In [58]:
inputs = {
    "extract": results["extract"]
}

func = eval(tasks["transform"]["func"])
results["transform"] = func(inputs)

print(results)

{'extract': [1, 2, 3], 'transform': [2, 4, 6]}


In [59]:
inputs = {
    "transform": results["transform"]
}

func = eval(tasks["load"]["func"])
results["load"] = func(inputs)

print(results)

{'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}


In [60]:
tasks = {
    "load": {
        "deps": ["transform"],
        "func": "lambda inputs: sum(inputs['transform'])"
    },
    "extract": {
        "deps": [],
        "func": "lambda inputs: [1, 2, 3]"
    },
    "transform": {
        "deps": ["extract"],
        "func": "lambda inputs: [x * 2 for x in inputs['extract']]"
    }
}

results = {}

# Run extract
inputs = {}
func = eval(tasks["extract"]["func"])
results["extract"] = func(inputs)

# Run transform
inputs = {
    "extract": results["extract"]
}
func = eval(tasks["transform"]["func"])
results["transform"] = func(inputs)

# Run load
inputs = {
    "transform": results["transform"]
}
func = eval(tasks["load"]["func"])
results["load"] = func(inputs)

print(results)

{'extract': [1, 2, 3], 'transform': [2, 4, 6], 'load': 12}


In [61]:
def execute_dag(tasks):
    results = {}
    state = {}

    def run(task_name):
        if state.get(task_name) == 'visiting':       # loop? error
            raise Exception("Cycle detected")
        if state.get(task_name) == 'done':           # already done? reuse
            return results[task_name]

        state[task_name] = 'visiting'                # mark: starting

        inputs = {}
        for dep in tasks[task_name]['deps']:         # run deps first
            inputs[dep] = run(dep)

        func = eval(tasks[task_name]['func'])        # make the function
        results[task_name] = func(inputs)            # run it, save answer
        state[task_name] = 'done'                    # mark: finished
        return results[task_name]

    for task_name in tasks:
        run(task_name)
    return results

In [62]:
def execute_dag(tasks):
    results = {}
    done = set()
    visiting = set()

    def run(name):
        if name in visiting:
            raise Exception("Cycle!")
        if name in done:
            return results[name]

        visiting.add(name)
        inputs = {}
        for dep in tasks[name]['deps']:
            inputs[dep] = run(dep)

        # inline: turn the string into a function right here
        f = tasks[name]['func']
        if callable(f):
            func = f
        else:
            ns = {}
            exec(f"_f = {f}", ns)
            func = ns['_f']

        results[name] = func(inputs)

        visiting.remove(name)
        done.add(name)
        return results[name]

    for name in tasks:
        run(name)
    return results

In [63]:
import ast
import types

def execute_dag(tasks):
    results = {}
    done = set()
    visiting = set()

    def make_func(f):
        if callable(f):
            return f
        # parse the string into an AST, compile it, build a function object
        module = ast.parse(f, mode='eval')
        code_obj = compile(module, '<string>', 'eval')
        # the compiled lambda's code is the first constant in the code object
        lambda_code = code_obj.co_consts[0]
        return types.FunctionType(lambda_code, {})

    def run(name):
        if name in visiting:
            raise Exception("Cycle!")
        if name in done:
            return results[name]

        visiting.add(name)
        inputs = {}
        for dep in tasks[name]['deps']:
            inputs[dep] = run(dep)

        func = make_func(tasks[name]['func'])
        results[name] = func(inputs)

        visiting.remove(name)
        done.add(name)
        return results[name]

    for name in tasks:
        run(name)
    return results